# StyleBot — Entrega 3: integración CNN y aprendizaje por refuerzo

Este notebook reúne la versión final y reproducible de la **Entrega 3**. Integra el entorno de almacén, las políticas **Q-Learning** y **Dyna-Q**, la percepción mediante la CNN congelada, la evaluación común y la demostración de 50 prendas.

**Equipo y responsabilidades**

- **Krysto Alberto Adán Jauregui Zúñiga:** coordinación, datos, protocolo y validación final.
- **Luis Ángel Mendoza Gaytan:** entorno, Q-Learning y Dyna-Q.
- **Juan José Aranda Flores:** CNN e integración entre percepción y entorno.
- **Ángel Uriel Martínez Godínez:** métricas, comparación y análisis de fallos.
- **Jacqueline Soto:** integración, documentación y demostración.

Los experimentos detallados permanecen en `01_qlearning_dynaq.ipynb`, `02_integracion_cnn_rl.ipynb` y `03_evaluacion_rl.ipynb`. Este archivo carga sus resultados publicados, los compara y comprueba que la entrega esté completa sin volver a entrenar los modelos.

In [ ]:
# 1. Preparación reproducible en Google Colab
from pathlib import Path
import json
import os
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gymnasium"])

REPO_URL = "https://github.com/krysto9872/StyleBot.git"
REPO = Path("/content/StyleBot")

if REPO.exists():
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only"])
else:
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO)])

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

print("Repositorio preparado en:", Path.cwd())
print("Commit utilizado:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

## 2. Protocolo común y definición del problema

StyleBot recibe prendas en la zona de recepción de un almacén de $8\times8$. Una CNN congelada clasifica cada imagen de Fashion-MNIST y proporciona la categoría percibida. Después, un agente decide cómo mover el robot y en qué estante colocar la prenda.

El proceso se modela como un **MDP**:

- **Estado:** posición del robot, categoría percibida de la carga y condición del estante objetivo.
- **Acciones:** arriba, abajo, izquierda, derecha, tomar y colocar.
- **Recompensas:** premian una colocación correcta y penalizan pasos, acciones inválidas, estantes llenos y colocaciones incorrectas.
- **Terminación:** finaliza cuando se procesan las prendas del episodio o se alcanza el máximo de pasos.

La política nunca recibe la etiqueta real. Solo utiliza la predicción de la CNN; por eso los errores se separan entre **percepción CNN** y **decisión de la política**.

In [ ]:
# 3. Carga y presentación del protocolo oficial
RUTA_PROTOCOLO = Path("datos/entrega3/protocolo_entrega3.json")
with RUTA_PROTOCOLO.open(encoding="utf-8") as archivo:
    protocolo = json.load(archivo)

categorias = pd.DataFrame({
    "id": range(len(protocolo["categorias"])),
    "categoría": protocolo["categorias"],
    "estante": [tuple(protocolo["estantes"][str(i)]) for i in range(10)],
})
acciones = pd.DataFrame([
    {"id": int(indice), "acción": nombre}
    for indice, nombre in protocolo["acciones"].items()
]).sort_values("id")
recompensas = pd.DataFrame([
    {"evento": evento, "recompensa": valor}
    for evento, valor in protocolo["recompensas"].items()
])

print("Versión:", protocolo["version"])
print("Mapa:", protocolo["tamano_mapa"], "| Estados:", protocolo["numero_estados"], "| Acciones:", protocolo["numero_acciones"])
print("Semilla general:", protocolo["semilla"], "| Semilla demostración:", protocolo["semilla_demo"])
display(categorias)
display(acciones)
display(recompensas)

## 3. Validación del entorno

El entorno reutilizable está en `src/entrega3/entorno_almacen.py`. La siguiente celda confirma que el espacio contiene **1,408 estados** y **6 acciones**, y que puede reiniciarse con los mismos 50 ejemplos usados por la CNN.

In [ ]:
from src.entrega3.entorno_almacen import AlmacenStyleBot

predicciones = pd.read_csv("datos/entrega3/predicciones_50_prendas.csv")
entorno = AlmacenStyleBot(
    etiquetas=predicciones["etiqueta_real"].to_numpy(),
    predicciones=predicciones["prediccion"].to_numpy(),
    prendas_por_episodio=protocolo["prendas_demo"],
    capacidad=8,
    max_pasos=protocolo["prendas_demo"] * protocolo["max_pasos_por_prenda"],
    recompensas=protocolo["recompensas"],
    render_mode="ansi",
)

observacion, informacion = entorno.reset(
    seed=protocolo["semilla_demo"],
    options={"indices": np.arange(protocolo["prendas_demo"])},
)

assert entorno.observation_space.n == protocolo["numero_estados"]
assert entorno.action_space.n == protocolo["numero_acciones"]
print("Entorno reiniciado correctamente.")
print("Observación inicial:", observacion)
print("Espacio de estados:", entorno.observation_space.n)
print("Espacio de acciones:", entorno.action_space.n)
print(entorno.render())
entorno.close()

## 4. Entrenamiento y selección de políticas

Se compararon configuraciones de **Q-Learning** y **Dyna-Q**. La selección de hiperparámetros se realizó con validación y la evaluación final utilizó semillas separadas. Las tablas Q finales están almacenadas en `modelos/entrega3/` y no se vuelven a entrenar durante la evaluación.

In [ ]:
configuraciones = pd.read_csv("resultados/entrega3/configuraciones_hiperparametros.csv")
resumen_rl = pd.read_csv("resultados/entrega3/qlearning_dynaq_resumen.csv")

columnas_config = [
    columna for columna in [
        "agente", "configuracion", "alpha", "gamma", "epsilon_inicial",
        "epsilon_final", "episodios_entrenamiento", "recompensa_promedio",
        "tasa_exito_prendas", "estado"
    ] if columna in configuraciones.columns
]

print("Configuraciones evaluadas:", len(configuraciones))
display(configuraciones[columnas_config] if columnas_config else configuraciones)

columnas_resumen = [
    "agente", "configuracion", "episodios_entrenamiento",
    "recompensa_promedio", "tasa_exito_prendas", "pasos_promedio",
    "episodios_truncados", "estado"
]
display(resumen_rl[columnas_resumen])

In [ ]:
# Comparación visual de los tres agentes
comparacion = pd.read_csv("resultados/entrega3/comparacion_agentes.csv")
orden = ["Aleatorio", "Q-Learning", "Dyna-Q"]
comparacion["agente"] = pd.Categorical(comparacion["agente"], orden, ordered=True)
comparacion = comparacion.sort_values("agente")

fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
colores = ["#9ca3af", "#2563eb", "#16a34a"]

ejes[0].bar(comparacion["agente"], comparacion["tasa_exito_prendas"] * 100, color=colores)
ejes[0].set_title("Tasa de éxito por prendas")
ejes[0].set_ylabel("Éxito (%)")
ejes[0].set_ylim(0, 100)

ejes[1].bar(comparacion["agente"], comparacion["recompensa_promedio"], color=colores)
ejes[1].set_title("Recompensa promedio")
ejes[1].set_ylabel("Recompensa")

ejes[2].bar(comparacion["agente"], comparacion["pasos_promedio"], color=colores)
ejes[2].set_title("Pasos promedio")
ejes[2].set_ylabel("Pasos")

for eje in ejes:
    eje.tick_params(axis="x", rotation=20)
    eje.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()
display(comparacion[["agente", "recompensa_promedio", "tasa_exito_prendas", "pasos_promedio"]])

## 5. Integración de la CNN con el agente

La CNN se carga desde `modelos/stylebot_cnn_final.keras` y permanece congelada. Para la demostración se eligieron 50 imágenes con la semilla 2026. La categoría predicha determina el estante objetivo y la etiqueta real se conserva únicamente para medir el resultado.

In [ ]:
predicciones = pd.read_csv("datos/entrega3/predicciones_50_prendas.csv")
simulacion = pd.read_csv("resultados/entrega3/simulacion_50_prendas_rl.csv")

total = len(predicciones)
errores_cnn = int(predicciones["es_error"].astype(str).str.lower().eq("true").sum())
aciertos_cnn = total - errores_cnn

assert total == protocolo["prendas_demo"] == 50
print(f"CNN: {aciertos_cnn}/{total} clasificaciones correctas ({aciertos_cnn / total:.1%}).")
print(f"Errores de percepción: {errores_cnn}/{total}.")
display(predicciones.head(10))
display(simulacion[[
    "agente", "recompensa_total", "exito_prendas", "pasos",
    "correctas", "incorrectas", "fallos_cnn", "fallos_politica"
]])

In [ ]:
# Visualización de las imágenes que la CNN clasificó incorrectamente
from tensorflow.keras.datasets import fashion_mnist

(_, _), (imagenes_prueba, etiquetas_prueba) = fashion_mnist.load_data()
errores = predicciones[predicciones["es_error"].astype(str).str.lower().eq("true")].copy()

if errores.empty:
    print("No hubo errores CNN en las 50 prendas.")
else:
    fig, ejes = plt.subplots(1, len(errores), figsize=(4 * len(errores), 4))
    ejes = np.atleast_1d(ejes)
    for eje, (_, fila) in zip(ejes, errores.iterrows()):
        indice = int(fila["indice_test"])
        eje.imshow(imagenes_prueba[indice], cmap="gray")
        eje.set_title(
            f"Real: {fila['categoria_real']}\nPredicción: {fila['categoria_predicha']}\n"
            f"Confianza: {float(fila['confianza']):.3f}"
        )
        eje.axis("off")
    plt.suptitle("Errores de percepción CNN")
    plt.tight_layout()
    plt.show()
display(errores)

## 6. Evaluación común y separación de fallos

Los tres agentes fueron comparados con el mismo protocolo. Un **fallo CNN** ocurre cuando la categoría percibida no coincide con la real. Un **fallo de política** ocurre cuando, con una percepción correcta, el agente decide un estante incorrecto.

In [ ]:
fallos = pd.read_csv("resultados/entrega3/analisis_fallos.csv")
display(fallos)

tabla_fallos = fallos.pivot(index="agente", columns="tipo_fallo", values="casos").fillna(0)
tabla_fallos = tabla_fallos.reindex(["Aleatorio", "Q-Learning", "Dyna-Q"])
tabla_fallos.plot(kind="bar", figsize=(9, 4), color=["#f59e0b", "#dc2626"])
plt.title("Separación de fallos en 50 prendas")
plt.xlabel("Agente")
plt.ylabel("Número de casos")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

assert set(comparacion["agente"].astype(str)) == {"Aleatorio", "Q-Learning", "Dyna-Q"}

## 7. Demostración de 50 prendas

La demostración final utiliza la semilla 2026, las 50 predicciones congeladas y las tablas Q seleccionadas. La trayectoria contiene cada movimiento, acción y evento. El resumen esperado es **49 colocaciones correctas**, **1 error de CNN** y **0 fallos de política** para los agentes entrenados.

In [ ]:
trayectoria = pd.read_csv("resultados/entrega3/trayectoria_50_prendas_rl.csv")
eventos_entrega = trayectoria[trayectoria["evento"].isin(["correcto", "incorrecto", "devuelta"])]

print("Pasos registrados:", len(trayectoria))
print("Prendas procesadas:", len(eventos_entrega))
print("Eventos finales:")
display(eventos_entrega["evento"].value_counts().rename_axis("evento").to_frame("cantidad"))

fig, eje = plt.subplots(figsize=(8, 7))
dispersión = eje.scatter(
    trayectoria["columna"], trayectoria["fila"],
    c=trayectoria["paso"], cmap="viridis", s=18, alpha=0.65
)
eje.scatter([3, 4], [7, 7], marker="s", s=150, color="#ef4444", label="Recepción")
estantes = np.array(list(protocolo["estantes"].values()))
eje.scatter(estantes[:, 1], estantes[:, 0], marker="D", s=80, color="#f59e0b", label="Estantes")
eje.set_xlim(-0.5, 7.5)
eje.set_ylim(7.5, -0.5)
eje.set_xticks(range(8))
eje.set_yticks(range(8))
eje.grid(alpha=0.3)
eje.set_title("Trayectoria de la demostración de 50 prendas")
eje.set_xlabel("Columna")
eje.set_ylabel("Fila")
eje.legend()
plt.colorbar(dispersión, ax=eje, label="Paso")
plt.show()

## 8. Conclusiones

- **Dyna-Q** obtuvo el mejor resultado global: recompensa promedio de **-27.80**, éxito por prendas de **93.6 %** y **78.64 pasos** promedio.
- **Q-Learning** alcanzó **92.4 %** de éxito por prendas, muy por encima del agente aleatorio (**1.6 %**).
- En la demostración de 50 prendas, la CNN acertó **49 de 50**. El único error fue un **Suéter** reconocido como **Camiseta/top**.
- Q-Learning y Dyna-Q registraron **0 fallos de política** en la demostración; el error restante se atribuye a la percepción.
- La separación de fallos permite mejorar cada módulo de forma independiente: la CNN para percepción y las políticas para navegación y colocación.

## Fuentes

- Sutton, R. S. y Barto, A. G. *Reinforcement Learning: An Introduction* (2.ª ed.). http://incompleteideas.net/book/the-book-2nd.html
- Watkins, C. J. C. H. y Dayan, P. (1992). *Q-learning*. https://www.gatsby.ucl.ac.uk/~dayan/papers/cjch.pdf
- Gymnasium. Documentación oficial: https://gymnasium.farama.org/
- TensorFlow. Fashion-MNIST: https://www.tensorflow.org/api_docs/python/tf/keras/datasets/fashion_mnist/load_data

In [ ]:
# 9. Comprobación automática de la entrega
archivos_requeridos = [
    "datos/entrega3/protocolo_entrega3.json",
    "datos/entrega3/predicciones_50_prendas.csv",
    "src/entrega3/entorno_almacen.py",
    "src/entrega3/agentes_rl.py",
    "src/entrega3/percepcion_cnn.py",
    "modelos/stylebot_cnn_final.keras",
    "modelos/entrega3/q_table_qlearning.npz",
    "modelos/entrega3/q_table_dynaq.npz",
    "resultados/entrega3/configuraciones_hiperparametros.csv",
    "resultados/entrega3/comparacion_agentes.csv",
    "resultados/entrega3/analisis_fallos.csv",
    "resultados/entrega3/simulacion_50_prendas_rl.csv",
    "resultados/entrega3/trayectoria_50_prendas_rl.csv",
]

faltantes = [ruta for ruta in archivos_requeridos if not Path(ruta).exists()]
assert not faltantes, "Faltan archivos: " + ", ".join(faltantes)
assert len(predicciones) == 50
assert errores_cnn == 1
assert set(comparacion["agente"].astype(str)) == {"Aleatorio", "Q-Learning", "Dyna-Q"}
assert Path("modelos/entrega3/q_table_qlearning.npz").stat().st_size > 0
assert Path("modelos/entrega3/q_table_dynaq.npz").stat().st_size > 0

print("✅ VALIDACIÓN FINAL COMPLETA")
print("✅ Todos los archivos técnicos requeridos existen.")
print("✅ Se cargaron 50 prendas y se separaron los errores CNN/política.")
print("✅ Se compararon Aleatorio, Q-Learning y Dyna-Q.")
print("El notebook está listo para guardarse con todas sus salidas.")